# Decision Tree Classifier (CART) — Solution

**Difficulty**: 🟡 Medium

---

### Problem Statement

Decision trees are the weak learners inside random forests and gradient-boosted trees
(XGBoost, LightGBM, CatBoost), which are still the default models for tabular data. They also
come up constantly in ML-breadth interviews. **CART** (Breiman et al., 1984) grows a binary tree
greedily. At every node it tries each feature $f$ and threshold $t$, sends samples with
$x_f \le t$ left and the rest right, and keeps the split whose children are purest.

For a node whose samples have class proportions $p = (p_1, \dots, p_C)$, the **Gini impurity** is

$$G(p) = 1 - \sum_{c=1}^{C} p_c^2 .$$

A split of $n$ samples into $n_L$ left and $n_R$ right is scored by the **weighted child impurity**

$$\frac{n_L}{n}\,G(p_L) + \frac{n_R}{n}\,G(p_R),$$

and CART picks the split that minimizes it. Writing that loop is easy. What interviewers probe
is the **split search**: recomputing the impurity from scratch for every threshold costs
$O(n^2)$ per feature. Sorting the column once and using cumulative class counts scores *every*
threshold in one vectorized pass, in $O(n \log n)$.

---

### Requirements

1. **`DecisionTreeClassifier(max_depth=None, min_samples_split=2)`**
   - **`fit(X, y)`**: `X` is a `(N, D)` float tensor and `y` a `(N,)` integer tensor of labels in
     `[0, C)` with **`C = y.max() + 1`**. Returns `self`.
   - **`predict_proba(X)`** → `(M, C)`: the class distribution stored in the leaf each row falls into.
   - **`predict(X)`** → `(M,)`: the most probable class (argmax of `predict_proba`).
   - **`get_depth()`** → `int`: depth of the fitted tree. The root is depth 0, so a tree that is a
     single leaf has depth 0.
2. **Candidate thresholds** for feature $f$ at a node are the **midpoints between consecutive
   distinct sorted values** of $x_f$ among that node's samples. A row goes **left iff
   `x[f] <= threshold`**. Never cut between two equal values.
3. **Split choice**: the (feature, threshold) with the lowest weighted child Gini over all features.
   Ties may be broken any way.
4. **Stopping**: a node becomes a leaf if
   - its depth equals `max_depth` (`None` means unlimited), or
   - it has fewer than `min_samples_split` samples, or
   - no candidate split gives a weighted child impurity **strictly lower** than the node's own
     impurity. This covers pure nodes and nodes whose rows are all identical.
5. **Leaves** store the **class distribution**: the fraction of the node's training samples in each
   of the `C` classes, not just the majority class.

---

### Constraints

- ✅ PyTorch tensor ops. Python loops over *nodes* are fine.
- ✅ Score all thresholds of a feature with one sort + cumulative counts, not a loop over thresholds.
- ❌ No scikit-learn or other ML libraries.
- ❌ Don't recompute the impurity from scratch for each candidate threshold ($O(n^2)$ per feature).
  It gives the right answer, but it's the version an interviewer pushes back on.

---

<details>
  <summary>💡 Hint</summary>

  - Sort the node's samples by one feature. A cut between sorted positions $i$ and $i+1$ sends
    exactly the first $i+1$ samples left. How do the class counts on the left change as $i$ grows?
  - Turn the labels into a one-hot matrix `(n, C)`. Its cumulative sum along the sorted order gives
    the left counts for *every* cut at once, and right = total − left.
  - With counts $c$ (and $n = \sum_c c_c$), $n \cdot G = n - \lVert c\rVert^2 / n$. Rewrite the
    weighted child impurity with this identity, then pick the cut with an `argmax`/`argmin`.
  - `X.sort(dim=0)` sorts every column at once, so you can score all features in a single pass.
  - Positions where two neighbouring sorted values are equal are not real cuts. Mask them out.
  - Recursion is natural for growing the tree. For predicting, think about how to move *all* rows
    down one level at a time.

</details>

In [ ]:
import math

import torch
import torch.nn.functional as F

In [ ]:
# Three overlapping Gaussian blobs in 2D: a 3-class problem no single split can solve.
torch.manual_seed(0)
centers = torch.tensor([[0.0, 0.0], [2.5, 0.5], [1.0, 2.5]])
X = torch.cat([c + 0.8 * torch.randn(70, 2) for c in centers])     # (210, 2)
y = torch.arange(3).repeat_interleave(70)                           # (210,) labels 0, 1, 2
perm = torch.randperm(len(y))
X, y = X[perm], y[perm]
X_train, y_train, X_test, y_test = X[:150], y[:150], X[150:], y[150:]
print("X_train", tuple(X_train.shape), "class counts", y_train.bincount().tolist())

In [ ]:
class DecisionTreeClassifier:
    """CART classification tree: binary axis-aligned splits chosen by Gini impurity."""

    def __init__(self, max_depth=None, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split

    # ------------------------------------------------------------------ fit
    def fit(self, X, y):
        """
        Args:
            X: (N, D) float features.
            y: (N,) integer class labels in [0, C), with C = y.max() + 1.
        Returns:
            self
        """
        X = torch.as_tensor(X)
        X = X if X.is_floating_point() else X.float()
        y = torch.as_tensor(y).long()
        self.n_classes_ = int(y.max()) + 1
        Y = F.one_hot(y, self.n_classes_).double()          # (N, C) class indicators

        # Flat node arrays (like sklearn's tree_): node i is a leaf iff feature[i] == -1.
        feature, threshold, left, right, value = [], [], [], [], []
        self.depth_ = 0

        def new_node(idx, depth):
            counts = Y[idx].sum(0)                           # (C,) class counts at this node
            value.append(counts / counts.sum())              # leaf = class distribution
            feature.append(-1); threshold.append(0.0); left.append(-1); right.append(-1)
            self.depth_ = max(self.depth_, depth)
            return len(value) - 1

        # Iterative depth-first growth (no recursion limit on deep trees).
        root = new_node(torch.arange(X.shape[0]), 0)
        stack = [(root, torch.arange(X.shape[0]), 0)]
        while stack:
            node, idx, depth = stack.pop()
            if self.max_depth is not None and depth >= self.max_depth:
                continue
            if len(idx) < max(self.min_samples_split, 2):
                continue
            split = best_gini_split(X[idx], Y[idx])
            if split is None:                                # pure, constant, or no gain
                continue
            f, t = split
            go_left = X[idx, f] <= t
            feature[node], threshold[node] = f, t
            left[node] = new_node(idx[go_left], depth + 1)
            right[node] = new_node(idx[~go_left], depth + 1)
            stack.append((left[node], idx[go_left], depth + 1))
            stack.append((right[node], idx[~go_left], depth + 1))

        self.feature_ = torch.tensor(feature)
        self.threshold_ = torch.tensor(threshold, dtype=X.dtype)
        self.left_ = torch.tensor(left)
        self.right_ = torch.tensor(right)
        self.value_ = torch.stack(value).to(X.dtype)         # (n_nodes, C)
        return self

    def get_depth(self):
        """Depth of the fitted tree; a tree that is a single leaf has depth 0."""
        return self.depth_

    # -------------------------------------------------------------- predict
    def predict_proba(self, X):
        """(M, D) -> (M, C): class distribution of the leaf each row lands in."""
        X = torch.as_tensor(X, dtype=self.threshold_.dtype)
        rows = torch.arange(X.shape[0])
        node = torch.zeros(X.shape[0], dtype=torch.long)    # every row starts at the root
        for _ in range(self.depth_):                        # all rows descend one level per step
            f = self.feature_[node]
            go_left = X[rows, f.clamp(min=0)] <= self.threshold_[node]
            child = torch.where(go_left, self.left_[node], self.right_[node])
            node = torch.where(f < 0, node, child)          # rows already at a leaf stay put
        return self.value_[node]

    def predict(self, X):
        """(M, D) -> (M,) most probable class."""
        return self.predict_proba(X).argmax(dim=1)


def best_gini_split(X, Y):
    """Best (feature, threshold) for one node, or None if no split lowers the impurity.

    Scores every candidate threshold of every feature in one vectorized pass:
    sort each column once, then class counts left of every cut are a cumsum.

    Args:
        X: (n, D) features of the samples at this node.
        Y: (n, C) one-hot labels (float64) of the same samples.
    """
    n, D = X.shape
    xs, order = X.sort(dim=0)                       # (n, D): each column sorted independently
    counts_left = Y[order].cumsum(dim=0)[:-1]       # (n-1, D, C): cut after sorted position i
    counts_right = Y.sum(0) - counts_left           # (n-1, D, C)
    n_left = torch.arange(1, n, dtype=Y.dtype)[:, None]   # (n-1, 1)
    n_right = n - n_left

    # n_L*Gini_L + n_R*Gini_R = n - (|c_L|^2 / n_L + |c_R|^2 / n_R), so minimizing the
    # weighted child impurity is the same as maximizing this score.
    score = (counts_left.pow(2).sum(-1) / n_left
             + counts_right.pow(2).sum(-1) / n_right)          # (n-1, D)
    # A threshold can only fall between two *distinct* values of the feature.
    score = score.masked_fill(xs[1:] == xs[:-1], float("-inf"))

    best = int(score.argmax())
    i, f = divmod(best, D)
    parent_score = Y.sum(0).pow(2).sum() / n        # the same quantity with no split
    if not score[i, f] - parent_score > 1e-9:       # require a strict impurity decrease
        return None
    threshold = (xs[i, f] + xs[i + 1, f]) / 2       # midpoint between consecutive values
    return f, float(threshold)

In [ ]:
# Test 1: a depth-1 tree is exactly the best single split, found here by brute force
def gini(labels, C):
    p = torch.bincount(labels, minlength=C).double() / len(labels)
    return 1 - (p ** 2).sum().item()

def brute_force_stump(X, y, C):
    best = (float("inf"), None, None)
    for f in range(X.shape[1]):
        vals = torch.unique(X[:, f])                          # sorted distinct values
        for t in ((vals[:-1] + vals[1:]) / 2).tolist():       # midpoints
            m = X[:, f] <= t
            wg = (m.sum().item() * gini(y[m], C) + (~m).sum().item() * gini(y[~m], C)) / len(y)
            if wg < best[0]:
                best = (wg, f, t)
    return best

best_wg, f, t = brute_force_stump(X_train, y_train, 3)
stump = DecisionTreeClassifier(max_depth=1).fit(X_train, y_train)
P = stump.predict_proba(X_train)
left = X_train[:, f] <= t
for side in (left, ~left):
    want = torch.bincount(y_train[side], minlength=3).float() / side.sum()
    assert torch.allclose(P[side], want.expand(int(side.sum()), 3), atol=1e-6), \
        "the depth-1 tree's leaves differ from the brute-force best stump"
print(f"best stump: feature {f}, threshold {t:.4f}, weighted Gini {best_wg:.4f}")
print("Test 1 passed: depth-1 tree == brute-force best stump")

In [ ]:
# Test 2: depth control, memorization, probabilities
deep = DecisionTreeClassifier().fit(X_train, y_train)
train_acc = (deep.predict(X_train) == y_train).float().mean().item()
assert train_acc == 1.0, f"an unlimited-depth tree should fit the training set, got {train_acc:.1%}"

for d in (1, 2, 3, 4):
    tree = DecisionTreeClassifier(max_depth=d).fit(X_train, y_train)
    assert tree.get_depth() == d, f"max_depth={d}: expected depth {d}, got {tree.get_depth()}"

P = deep.predict_proba(X_test)
assert P.shape == (len(X_test), 3), P.shape
assert torch.allclose(P.sum(1), torch.ones(len(X_test))), "rows of predict_proba must sum to 1"
assert torch.equal(deep.predict(X_test), P.argmax(1))

shallow = DecisionTreeClassifier(max_depth=3).fit(X_train, y_train)
test_acc = (shallow.predict(X_test) == y_test).float().mean().item()
print(f"max_depth=None: depth {deep.get_depth()}, train accuracy {train_acc:.1%}")
print(f"max_depth=3   : test accuracy {test_acc:.1%}")
assert test_acc > 0.6, f"test accuracy {test_acc:.1%} is unexpectedly low"
print("Test 2 passed")

In [ ]:
# Test 3: edge cases and XOR
pure = DecisionTreeClassifier().fit(X_train, torch.full((len(X_train),), 2))
assert pure.get_depth() == 0, "a pure node must stay a leaf"
assert torch.allclose(pure.predict_proba(X_test[:1]), torch.tensor([[0.0, 0.0, 1.0]]))

const = DecisionTreeClassifier().fit(torch.ones(6, 2), torch.tensor([0, 1, 1, 0, 1, 1]))
assert const.get_depth() == 0, "identical rows: there is no threshold to split on"
assert torch.allclose(const.predict_proba(torch.ones(1, 2)), torch.tensor([[1 / 3, 2 / 3]]))

# XOR: four corner clusters labelled (x0 > 0) xor (x1 > 0)
g = torch.Generator().manual_seed(3)
corners = [((-1.0, -1.0), 0, 16), ((1.0, 1.0), 0, 8), ((-1.0, 1.0), 1, 4), ((1.0, -1.0), 1, 12)]
X_xor = torch.cat([torch.tensor(c) + 0.2 * torch.randn(n, 2, generator=g) for c, _, n in corners])
y_xor = torch.cat([torch.full((n,), label) for _, label, n in corners])
acc = {d: (DecisionTreeClassifier(max_depth=d).fit(X_xor, y_xor).predict(X_xor) == y_xor).float().mean().item()
       for d in (1, 2)}
print(f"XOR accuracy: depth 1 = {acc[1]:.1%}, depth 2 = {acc[2]:.1%}")
assert acc[1] <= 0.75 and acc[2] == 1.0
print("Test 3 passed")

In [ ]:
# Auto-grade (one-time setup from the repo root: pip install -e python/)
from torchleet import check
check("decision-tree", DecisionTreeClassifier=DecisionTreeClassifier)

In [ ]:
# Optional: the vectorized split search vs. a per-threshold loop (same answer, very different cost)
import time

def loop_split(X, y, C):
    best = (float("inf"), None, None)
    for f in range(X.shape[1]):
        vals = torch.unique(X[:, f])
        for t in ((vals[:-1] + vals[1:]) / 2).tolist():
            m = X[:, f] <= t
            n_l, n_r = m.sum(), (~m).sum()
            g_l = 1 - ((torch.bincount(y[m], minlength=C) / n_l) ** 2).sum()
            g_r = 1 - ((torch.bincount(y[~m], minlength=C) / n_r) ** 2).sum()
            w = ((n_l * g_l + n_r * g_r) / len(y)).item()
            if w < best[0]:
                best = (w, f, t)
    return best[1:]

g = torch.Generator().manual_seed(0)
Xb = torch.randn(2000, 4, generator=g)
yb = ((Xb[:, 0] + 0.5 * torch.randn(2000, generator=g)) > 0).long()
t0 = time.time(); fast = best_gini_split(Xb, F.one_hot(yb, 2).double()); t1 = time.time()
slow = loop_split(Xb, yb, 2); t2 = time.time()
print(f"vectorized: {fast}  in {1e3 * (t1 - t0):.1f} ms")
print(f"loop      : {slow}  in {1e3 * (t2 - t1):.1f} ms")

## Why it works

**Greedy CART.** Finding the optimal tree is NP-hard, so CART makes the locally best split at each
node and recurses. Every node solves the same small problem: find the (feature, threshold)
with the lowest weighted child impurity.

**The Gini identity.** For a node with class counts $c$ and $n = \sum_k c_k$, $p = c/n$ gives
$n\,G(p) = n - \lVert c\rVert^2 / n$. Hence

$$n_L G_L + n_R G_R \;=\; n - \Big(\tfrac{\lVert c_L\rVert^2}{n_L} + \tfrac{\lVert c_R\rVert^2}{n_R}\Big),$$

so minimizing the weighted impurity is the same as maximizing the bracket. That bracket only
needs the left/right **counts** for each cut.

**One vectorized pass, $O(n \log n)$ per feature.** Sort the column (`X.sort(dim=0)` sorts all
$D$ columns at once). A cut after sorted position $i$ sends the first $i+1$ samples left, so the
left counts for *all* $n-1$ cuts are the prefix sums `Y[order].cumsum(0)` of the one-hot labels in
sorted order, and the right counts are `total − left`. The score of every cut is then one
elementwise expression on an `(n−1, D)` tensor. Cuts where `xs[i] == xs[i+1]` are masked to
$-\infty$: they don't correspond to any threshold, because `x <= t` must send equal values the
same way. The cost is the sort, $O(n \log n)$, plus $O(nC)$ for the counts. The naive
loop over thresholds re-scans all $n$ samples for each of the $n-1$ thresholds, which is
$O(n^2)$. The optional demo cell shows the difference.

**Strict decrease.** $G$ is strictly concave, and expanding the squares gives
$$\tfrac{n_L}{n}G(p_L) + \tfrac{n_R}{n}G(p_R) = G(p) - \tfrac{n_L n_R}{n^2}\,\lVert p_L - p_R\rVert^2 .$$
So a split lowers the impurity **iff the children's class distributions differ**. Pure nodes
and all-identical rows can never be split. With continuous features an impure node can always be
split (peel off the smallest value), so an unlimited tree memorizes any data without conflicting
duplicates. This identity also explains the XOR failure in follow-up 4.

**Vectorized prediction.** The tree is stored as flat arrays (`feature_`, `threshold_`, `left_`,
`right_`, `value_`), like sklearn's `tree_`. All rows start at the root and every iteration moves
each row one level down with a gather plus `torch.where`. That is `depth` tensor ops in total,
instead of a Python loop per row. Growth uses an explicit stack, so deep trees don't hit Python's
recursion limit.

## Complexity

- **Fit:** each tree level partitions the $N$ samples, so it costs $O(D \cdot N \log N + D N C)$ per
  level and $O(\text{depth} \cdot D N \log N)$ overall. Memory per node is $O(n D C)$ for the count tensor.
- **Predict:** $O(M \cdot \text{depth})$.
- Re-sorting at every node can be avoided: sort each feature once and keep the order when
  partitioning (that's what sklearn's presort did), or use histograms (follow-up 6).

## Common bugs

- **Cutting between equal values.** The counts assume the first $i+1$ samples go left, but `x <= t`
  sends every copy of the value left. The scored partition isn't the real one, and you can create
  an empty child that recurses forever.
- **Unweighted average** $(G_L + G_R)/2$. A 1-sample pure child then looks as good as a 1000-sample one.
- **Threshold at a data value** instead of the midpoint, or mixing `<` and `<=` between fit and
  predict. Unseen points between the two values go the wrong way.
- **Off-by-one** in depth (the root is depth 0) or `min_samples_split` (`n < min_samples_split` is a leaf).
- **Computing `C` per node** (e.g. `y.unique()`) instead of once from the full `y`. The
  probability columns stop lining up across leaves.
- **Leaves that store the majority class only.** `predict_proba` becomes one-hot, and calibration and
  ensembling (random forests average probabilities) suffer.
- **No stopping rule for zero gain** in an implementation that recomputes partitions. It can split forever.

## Interview follow-ups

1. **Gini vs. entropy?** Both are strictly concave impurities that are 0 on pure nodes. Gini is the
   error rate of guessing a label drawn from the node's distribution; entropy gives information
   gain. They choose the same split in the vast majority of cases, and Gini avoids the `log`.
   Misclassification error is *not* strictly concave, so it can report zero gain for a useful split.
   That's why it isn't used for growing.
2. **How do you stop a tree from overfitting?** Pre-pruning: `max_depth`, `min_samples_leaf`,
   `min_impurity_decrease`. Post-pruning: cost-complexity pruning minimizes
   $R(T) + \alpha\,|\text{leaves}(T)|$ along a nested sequence of subtrees, with $\alpha$ chosen by
   cross-validation (sklearn's `ccp_alpha`). Alternatively, average many deep trees (bagging /
   random forests) to cut variance.
3. **How is feature importance computed, and what's wrong with it?** Mean decrease in impurity:
   sum $n_\text{node}\,\Delta G$ over the nodes that split on $f$, then normalize. It is
   computed on training data and favours high-cardinality / continuous features, which have more
   thresholds to overfit with. Permutation importance on held-out data or SHAP values are less biased.
4. **Why does a greedy tree with a strict-gain rule fail on perfectly balanced XOR?** Every
   axis-aligned split leaves both children 50/50, so by the identity above the gain is exactly 0
   and the root stays a leaf. scikit-learn still splits at zero gain (its `min_impurity_decrease=0`
   test is not strict), and randomized trees or lookahead also escape. The test data here uses
   unequal cluster sizes so a strictly-positive greedy split exists.
5. **Categorical features and missing values?** For binary classification, sort the $K$ categories
   by their positive rate and treat them as ordered. Breiman showed the best of the $2^{K-1}$
   partitions is one of the $K-1$ cuts in that order. For missing values, CART uses surrogate
   splits; XGBoost/LightGBM learn a default direction per split.
6. **How do XGBoost/LightGBM make split search fast?** Histogram binning. Each feature is quantized
   into ≤ 256 bins once, a node accumulates per-bin label (or gradient) sums in $O(n)$, cuts are
   scored over bins in $O(\text{bins})$, and a child's histogram is the parent's minus its sibling's.